# Quick Start Guide
github link: [cytozip](https://github.com/DingWB/cytozip)

## 1. Installation

### conda
https://anaconda.org/bioconda/cytozip
```shell
conda install -c bioconda cytozip
# or
mamba install -c bioconda cytozip
```

### pip
```shell
# Prerequisites (one of):
#   conda install -c bioconda htslib libdeflate          # recommended
#   apt-get install libhts-dev libdeflate-dev            # Debian/Ubuntu
#   brew install htslib libdeflate                       # macOS
pip install cytozip
# or reinstall the latest version from github
pip uninstall -y cytozip && pip install git+http://github.com/DingWB/cytozip
```

In [1]:
# Make sure czip is installed:
!which czip

~/Software/conda/m3c/bin/czip


In [2]:
import os
os.chdir(os.path.expanduser("~/Projects/test_cytozip"))

## 2. Build reference cz file from a reference genome fasta file
The reference holds the genome-wide (chrom, pos, strand, context) axis. Per-cell .cz then store only mc/cov and reuse the reference’s positions, cutting per-cell size by ~5×.

In [2]:
!time czip build_ref --genome ~/Ref/hg38/hg38_ucsc_with_chrL.fa \ # reference genome 
    --output ~/Ref/hg38/hg38_with_chrL.allc.cz  \ # output reference cz file 
    --jobs 24 \ # number of CPU 
    --chrom_size ~/Ref/hg38/hg38_ucsc_with_chrL.main.chrom.sizes \ # restrited to main chromosomes (optional)

2026-07-22 11:34:57.492 | DEBUG    | cytozip.allc:WriteC:66 - chr1
2026-07-22 11:34:58.126 | DEBUG    | cytozip.allc:WriteC:66 - chr10
2026-07-22 11:34:58.860 | DEBUG    | cytozip.allc:WriteC:66 - chr11
2026-07-22 11:34:59.571 | DEBUG    | cytozip.allc:WriteC:66 - chr12
2026-07-22 11:35:00.213 | DEBUG    | cytozip.allc:WriteC:66 - chr13
2026-07-22 11:35:00.779 | DEBUG    | cytozip.allc:WriteC:66 - chr14
2026-07-22 11:35:01.325 | DEBUG    | cytozip.allc:WriteC:66 - chr15
2026-07-22 11:35:01.821 | DEBUG    | cytozip.allc:WriteC:66 - chr16
2026-07-22 11:35:02.279 | DEBUG    | cytozip.allc:WriteC:66 - chr17
2026-07-22 11:35:02.756 | DEBUG    | cytozip.allc:WriteC:66 - chr18
2026-07-22 11:35:03.066 | DEBUG    | cytozip.allc:WriteC:66 - chr19
2026-07-22 11:35:04.575 | DEBUG    | cytozip.allc:WriteC:66 - chr2
2026-07-22 11:35:04.794 | DEBUG    | cytozip.allc:WriteC:66 - chr20
2026-07-22 11:35:05.038 | DEBUG    | cytozip.allc:WriteC:66 - chr21
2026-07-22 11:35:05.320 | DEBUG    | cytozip.allc:

In [3]:
!czip header -I ~/Ref/hg38/hg38_with_chrL.allc.cz

magic  :  b'CZIP'
version  :  0.35
total_size  :  1430352768
message  :  /home/x-wding2/Ref/hg38/hg38_ucsc_with_chrL.fa
formats  :  ['Q', 'c', '3s']
columns  :  ['pos', 'strand', 'context']
sort_col  :  0
delta_cols  :  [0]
chunk_dims  :  ['chrom']
header_size  :  102


In [5]:
! czip view -I ~/Ref/hg38/hg38_with_chrL.allc.cz --show_dims 0 | head

chrom	pos	strand	context
chr1	10004	+	CCC
chr1	10005	+	CCT
chr1	10006	+	CTA
chr1	10010	+	CCC
chr1	10011	+	CCT
chr1	10012	+	CTA
chr1	10016	+	CCC
chr1	10017	+	CCT
chr1	10018	+	CTA


## 3. Call methylation from bam file and save as .cz format

### 2.1 Download example bam files

```shell
# Download example bam files from figshare with your browser: https://figshare.com/ndownloader/files/63998524
# or download with command line using pyfigshare:
pip install pyfigshare # https://github.com/DingWB/pyfigshare
figshare download 32095567 --file_id 66902144,66902129,66902816,66902822 --outdir cytozip_example_data
```

### 2.2 Call DNA methylation using cytozip

In [10]:
! time czip bam_to_cz --input cytozip_example_data/hg38_bam/UWA7648_CX1819_NAC_1_P10-1-K18-A10.bam \
               --genome ~/Ref/hg38/hg38_ucsc_with_chrL.fa \
               --output cytozip_example_data/hg38_bam/UWA7648_CX1819_NAC_1_P10-1-K18-A10.cz \
               --reference ~/Ref/hg38/hg38_with_chrL.allc.cz

[W::hts_idx_load3] The index file is older than the data file: cytozip_example_data/hg38_bam/UWA7648_CX1819_NAC_1_P10-1-K18-A10.bam.bai

real	1m33.907s
user	1m21.110s
sys	0m9.285s


Python API
```python
from cytozip.bam import bam_to_cz
bam_to_cz(bam_path="cytozip_example_data/hg38_bam/UWA7648_CX1819_NAC_1_P10-1-K18-A10.bam", \
                  genome="~/Ref/hg38/hg38_ucsc_with_chrL.fa", \
                  output="cytozip_example_data/hg38_bam/UWA7648_CX1819_NAC_1_P10-1-K18-A10.cz", \
                  reference="~/Ref/hg38/hg38_with_chrL.allc.cz")
```

## 4. Convert allc/bed/stdin to .cz

### 4.1 convert stream input to cz with pipe

In [3]:
! czip tocz -h

usage: czip tocz [-h] -O OUTPUT [-I INPUT] [-F FORMATS] [-C COLUMNS]
                 [-D CHUNK_DIMS] [-u USECOLS] [-d KEY_COLS] [-s SEP]
                 [-c BATCH_SIZE] [--header HEADER] [--skiprows SKIPROWS]
                 [-m MESSAGE] [-l LEVEL] [--delta_cols DELTA_COLS]

options:
  -h, --help            show this help message and exit
  -O OUTPUT, --output OUTPUT
                        output .cz file (default: None)
  -I INPUT, --input INPUT
                        input file (stdin if omitted) (default: None)
  -F FORMATS, --formats FORMATS
                        column formats, comma-separated (default: ['B', 'B'])
  -C COLUMNS, --columns COLUMNS
                        column names, comma-separated (default: ['mc', 'cov'])
  -D CHUNK_DIMS, --chunk_dims CHUNK_DIMS
                        chunk-key (dimension) names, comma-separated (default:
                        ['chrom'])
  -u USECOLS, --usecols USECOLS
                        column indices to pack, comma-separated (de

In [4]:
! zcat cytozip_example_data/hg38_allc/UWA7648_CX1819_NAC_1_P10-1-K18-A10.allc.tsv.gz | head

chr1	14932	-	CTT	0	1	1
chr1	14933	-	CCT	0	1	1
chr1	14935	-	CAC	1	1	1
chr1	14938	-	CAG	1	1	1
chr1	14939	-	CCA	0	1	1
chr1	14944	-	CTG	0	1	1
chr1	14945	-	CCT	0	1	1
chr1	14946	-	CCC	0	1	1
chr1	14948	-	CGC	1	1	1
chr1	14949	-	CCG	0	1	1

gzip: stdout: Broken pipe


In [10]:
# write chrom, position, mc and cov into .cz file with DEFLATE compress
! zcat cytozip_example_data/hg38_allc/UWA7648_CX1819_NAC_1_P10-1-K18-A10.allc.tsv.gz | czip tocz \
        --output cytozip_example_data/hg38_allc/UWA7648_CX1819_NAC_1_P10-1-K18-A10.cz \
        --formats Q,B,B --usecols 1,4,5 --columns pos,mc,cov \
        --delta_cols pos # file size would be smaller with DEFLATE compress: 47M vs 29M

In [11]:
! czip view -I cytozip_example_data/hg38_allc/UWA7648_CX1819_NAC_1_P10-1-K18-A10.cz --show_dims 0 | head

chrom	pos	mc	cov
chr1	14932	0	1
chr1	14933	0	1
chr1	14935	1	1
chr1	14938	1	1
chr1	14939	0	1
chr1	14944	0	1
chr1	14945	0	1
chr1	14946	0	1
chr1	14948	1	1


### 4.2 allc to cz with reference cz as input

In [12]:
! czip allc2cz -h

usage: czip allc2cz [-h] -I INPUT -O OUTPUT [-r REFERENCE]
                    [--missing_value MISSING_VALUE] [-F FORMATS] [-C COLUMNS]
                    [-D CHUNK_DIMS] [-u USECOLS] [--ref_pos_col REF_POS_COL]
                    [--allc_pos_col ALLC_POS_COL] [-s SEP]
                    [--chrom_order CHROM_ORDER] [-c BATCH_SIZE]
                    [--sort_col SORT_COL] [--delta_cols DELTA_COLS] [-j JOBS]
                    [--pattern PATTERN] [--no_skip_existing]

options:
  -h, --help            show this help message and exit
  -I INPUT, --input INPUT
                        input allc.tsv.gz, OR a directory containing many
                        allc.tsv.gz (batch mode: --output must be a directory)
                        (default: None)
  -O OUTPUT, --output OUTPUT
                        output .cz file (single-file), or output directory
                        (batch mode) (default: None)
  -r REFERENCE, --reference REFERENCE
                        reference .cz file (

In [14]:
! rm -f cytozip_example_data/hg38_allc/UWA7648_CX1819_NAC_1_P10-1-K18-A10.cz
! time czip allc2cz --input cytozip_example_data/hg38_allc/UWA7648_CX1819_NAC_1_P10-1-K18-A10.allc.tsv.gz \
                    --output cytozip_example_data/hg38_allc/UWA7648_CX1819_NAC_1_P10-1-K18-A10.cz \
                    --reference ~/Ref/hg38/hg38_with_chrL.allc.cz \
                    --formats "B,B" # for single cell, B is fine (max is 255), please change to H for pseudobulk data

2026-07-22 14:30:42.028 | INFO     | cytozip.allc:allc2cz:348 - /anvil/projects/x-mcb130189/Wubin/test_cytozip/cytozip_example_data/hg38_allc/UWA7648_CX1819_NAC_1_P10-1-K18-A10.allc.tsv.gz

real	0m42.276s
user	0m35.840s
sys	0m4.946s


In [16]:
! ls cytozip_example_data/hg38_allc -sh

total 121M
 97M UWA7648_CX1819_NAC_1_P10-1-K18-A10.allc.tsv.gz
2.0M UWA7648_CX1819_NAC_1_P10-1-K18-A10.allc.tsv.gz.tbi
 22M UWA7648_CX1819_NAC_1_P10-1-K18-A10.cz


In [19]:
# view
! czip view -I cytozip_example_data/hg38_allc/UWA7648_CX1819_NAC_1_P10-1-K18-A10.cz --show_dims 0 | head

chrom	mc	cov
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0


In [21]:
# view cz file with coordinates (reference cz)
! czip view -I cytozip_example_data/hg38_allc/UWA7648_CX1819_NAC_1_P10-1-K18-A10.cz \
            --show_dims 0 \
            -r ~/Ref/hg38/hg38_with_chrL.allc.cz | head

chrom	pos	strand	context	mc	cov
chr1	10004	+	CCC	0	0
chr1	10005	+	CCT	0	0
chr1	10006	+	CTA	0	0
chr1	10010	+	CCC	0	0
chr1	10011	+	CCT	0	0
chr1	10012	+	CTA	0	0
chr1	10016	+	CCC	0	0
chr1	10017	+	CCT	0	0
chr1	10018	+	CTA	0	0


In [27]:
# query cz file
! time czip query -I cytozip_example_data/hg38_allc/UWA7648_CX1819_NAC_1_P10-1-K18-A10.cz \
            --chunk_key chr1 --start 14932 --end 14949 \
            -r ~/Ref/hg38/hg38_with_chrL.allc.cz

chrom	pos	strand	context	mc	cov
chr1	14932	-	CTT	0	1
chr1	14933	-	CCT	0	1
chr1	14935	-	CAC	1	1
chr1	14936	+	CTG	0	0
chr1	14938	-	CAG	1	1
chr1	14939	-	CCA	0	1
chr1	14940	+	CCC	0	0
chr1	14941	+	CCA	0	0
chr1	14942	+	CAG	0	0
chr1	14944	-	CTG	0	1
chr1	14945	-	CCT	0	1
chr1	14946	-	CCC	0	1
chr1	14947	+	CGG	0	0
chr1	14948	-	CGC	1	1
chr1	14949	-	CCG	0	1

real	0m0.251s
user	0m0.136s
sys	0m0.097s


## 5. Python API

In [3]:
from cytozip import Reader
reader=Reader("cytozip_example_data/hg38_allc/UWA7648_CX1819_NAC_1_P10-1-K18-A10.cz")
print(reader.header)
df_chunks=reader.summary_chunks(printout=False)
print(df_chunks.head())

{'magic': b'CZIP', 'version': 0.36, 'total_size': 22810155, 'message': 'hg38_with_chrL.allc.cz', 'formats': ['B', 'B'], 'columns': ['mc', 'cov'], 'sort_col': None, 'delta_cols': [], 'chunk_dims': ['chrom'], 'header_size': 61}
           chrom  chunk_start_offset  chunk_size  chunk_tail_offset  \
chunk_dims                                                            
(chr1,)     chr1                  61     1906076            1912030   
(chr2,)     chr2             1912030     1912984            3830947   
(chr3,)     chr3             3830947     1541226            5376994   
(chr4,)     chr4             5376994     1443646            6825093   
(chr5,)     chr5             6825093     1404376            8233866   

            chunk_nblocks  chunk_nrows  
chunk_dims                              
(chr1,)               734     96166571  
(chr2,)               739     96769083  
(chr3,)               600     78577742  
(chr4,)               554     72568001  
(chr5,)               547     

In [4]:
reader.chunk_key2offset

{('chr1',): 61,
 ('chr2',): 1912030,
 ('chr3',): 3830947,
 ('chr4',): 5376994,
 ('chr5',): 6825093,
 ('chr6',): 8233866,
 ('chr7',): 9558673,
 ('chr8',): 10822796,
 ('chr9',): 11951699,
 ('chr10',): 12908810,
 ('chr11',): 14011574,
 ('chr12',): 15114383,
 ('chr13',): 16196205,
 ('chr14',): 17003787,
 ('chr15',): 17696549,
 ('chr16',): 18349412,
 ('chr17',): 19030001,
 ('chr18',): 19690263,
 ('chr19',): 20347873,
 ('chr20',): 20822410,
 ('chr21',): 21390273,
 ('chr22',): 21689571,
 ('chrX',): 22001315,
 ('chrY',): 22701140,
 ('chrM',): 22800785,
 ('chrL',): 22801381}

In [5]:
data=reader.chunk2numpy(dims=('chr1',))
data['f0'],data['f1'] # mc and cov

(array([0, 0, 0, ..., 0, 0, 0], dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], dtype=uint8))

In [6]:
df=reader.chunk2df(dims=('chr1',))
df.head()

,mc,cov
0,0,0
1,0,0
2,0,0
3,0,0
4,0,0


In [8]:
r=reader.query(chunk_key=('chr1',),start=14932,end=14949,printout=False,
                    reference="~/Ref/hg38/hg38_with_chrL.allc.cz")
#r.__next__()
for record in r:
    print(record)

['chr1', '14932', '-', 'CTT', '0', '1']
['chr1', '14933', '-', 'CCT', '0', '1']
['chr1', '14935', '-', 'CAC', '1', '1']
['chr1', '14936', '+', 'CTG', '0', '0']
['chr1', '14938', '-', 'CAG', '1', '1']
['chr1', '14939', '-', 'CCA', '0', '1']
['chr1', '14940', '+', 'CCC', '0', '0']
['chr1', '14941', '+', 'CCA', '0', '0']
['chr1', '14942', '+', 'CAG', '0', '0']
['chr1', '14944', '-', 'CTG', '0', '1']
['chr1', '14945', '-', 'CCT', '0', '1']
['chr1', '14946', '-', 'CCC', '0', '1']
['chr1', '14947', '+', 'CGG', '0', '0']
['chr1', '14948', '-', 'CGC', '1', '1']
['chr1', '14949', '-', 'CCG', '0', '1']


In [9]:
data=reader.query_numpy(chunk_key=('chr1',),start=14932,end=14949,
                    reference="~/Ref/hg38/hg38_with_chrL.allc.cz")
data

(array([14932, 14933, 14935, 14936, 14938, 14939, 14940, 14941, 14942,
        14944, 14945, 14946, 14947, 14948, 14949], dtype=uint64),
 array([(0, 1), (0, 1), (1, 1), (0, 0), (1, 1), (0, 1), (0, 0), (0, 0),
        (0, 0), (0, 1), (0, 1), (0, 1), (0, 0), (1, 1), (0, 1)],
       dtype=[('f0', 'u1'), ('f1', 'u1')]))

In [10]:
reader.close()

## 6. Remote Reading (from URL)
czip supports reading .cz files directly from HTTP/HTTPS URLs without downloading the entire file. This uses HTTP Range requests and a chunk index for O(1) lookup.

In [11]:
! time czip query -I https://neomorph.salk.edu/ftp/bican/UWA7648_CX1819_NAC_1_P10-1-K18-A10.cz \
            --chunk_key chr1 --start 14932 --end 14949 \
            -r https://neomorph.salk.edu/ftp/bican/hg38_with_chrL.allc.cz

chrom	pos	strand	context	mc	cov
chr1	14932	-	CTT	0	1
chr1	14933	-	CCT	0	1
chr1	14935	-	CAC	1	1
chr1	14936	+	CTG	0	0
chr1	14938	-	CAG	1	1
chr1	14939	-	CCA	0	1
chr1	14940	+	CCC	0	0
chr1	14941	+	CCA	0	0
chr1	14942	+	CAG	0	0
chr1	14944	-	CTG	0	1
chr1	14945	-	CCT	0	1
chr1	14946	-	CCC	0	1
chr1	14947	+	CGG	0	0
chr1	14948	-	CGC	1	1
chr1	14949	-	CCG	0	1

real	0m3.682s
user	0m0.264s
sys	0m0.177s


In [ ]:
import cytozip as czip

# Open a remote .cz file via URL
# url = "https://example.com/path/to/file.cz"
# reader = czip.Reader.from_url(url)
# reader.print_header()

# Or simply pass a URL to Reader (auto-detected)
# reader = czip.Reader(url)
# for record in reader.fetch(("chr1",)):
#     print(record)
# reader.close()
print("Remote reading requires a .cz file hosted on an HTTP server supporting Range requests.")